# 01 · Modern segmentation models

Run top to bottom. Existing Honours_Benchmark results are untouched. Start with the two-epoch smoke checks. Long training remains locked until they succeed.

Training and validation curves use separate splits. Test images never control patience or checkpoint selection.

## 1. Mount Drive

In [1]:
from pathlib import Path
from google.colab import drive
import sys, subprocess, json, os
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/SAR_Ship_Segmentation_Study')
assert (ROOT / 'sarstudy').is_dir(), f'Upload the whole project folder first: {ROOT}'
print('Project:', ROOT)


Mounted at /content/drive
Project: /content/drive/MyDrive/SAR_Ship_Segmentation_Study


In [ ]:
from pathlib import Path
import json
import zipfile

DRIVE = Path("/content/drive/MyDrive")
ZIP = DRIVE / "SAR_Study_Update_V5.zip"

assert ZIP.is_file(), f"Not found: {ZIP}"
print("ZIP size:", round(ZIP.stat().st_size / 1024, 1), "KB")

with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None, "The uploaded ZIP is corrupted"
    archive.extractall(DRIVE)

CONFIG = (
    DRIVE
    / "SAR_Ship_Segmentation_Study"
    / "configs"
    / "study.json"
)

settings = json.loads(CONFIG.read_text())

print("Experiment:", settings["experiment"])
print("Seed:", settings["seed"])

assert settings["experiment"] == "study_v5_leakage_safe"
assert settings["seed"] == 42

print("V5 update installed correctly")

ZIP size: 14.4 KB
Experiment: study_v5_leakage_safe
Seed: 42
V5 update installed correctly


In [2]:
from pathlib import Path
from google.colab import drive
import subprocess
import sys
import json
import os
from collections import deque

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/SAR_Ship_Segmentation_Study")
CONFIG = ROOT / "configs" / "study.json"

assert ROOT.is_dir(), ROOT
print("Project:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/SAR_Ship_Segmentation_Study


In [ ]:
print("run_final available:", "run_final" in globals())

run_final available: False


In [3]:
subprocess.run([
    sys.executable,
    str(ROOT / "scripts" / "setup_environment.py"),
    "modern",
], check=True)

PYTHON = Path("/content/sar-modern-env/bin/python")

assert PYTHON.is_file(), PYTHON
print("Modern environment ready:", PYTHON)

Modern environment ready: /content/sar-modern-env/bin/python


In [4]:
subprocess.run([
    str(PYTHON),
    "-m", "pip", "install",
    "--no-deps",
    "--force-reinstall",
    str(ROOT),
], check=True)

print("V5 project installed")

V5 project installed


## 2. Install packages

Run once per session and wait for READY. This installs packages, not training. No restart is needed. If installation fails, stop here and copy the error above the final traceback; do not run the training cells.

Use a CUDA GPU. This modern stack targets CUDA 12.8, including Blackwell; the preflight checks actual kernels. For comparable FPS across all models use the same GPU type, such as A100, in both training notebooks.

In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SAR_Ship_Segmentation_Study")
PYTHON = Path("/content/sar-modern-env/bin/python")

subprocess.run([
    str(PYTHON),
    "-m", "pip", "install",
    "--no-deps",
    "--force-reinstall",
    str(ROOT),
], check=True)

print("Updated project installed")

Updated project installed


## 3. ClearML credentials

In [5]:
from google.colab import userdata
# Secrets stay in this session's environment; they are never written into the package.
os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_SECRET_KEY')
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
print('ClearML credentials configured')


ClearML credentials configured


## 4. Frozen study settings

In [6]:
CONFIG = ROOT / 'configs' / 'study.json'
settings = json.loads(CONFIG.read_text())
print('Maximum epochs:', settings['max_epochs'], '| Patience:', settings['patience'])
print('Datasets:', settings['datasets'])
print('Experiment:', settings['experiment'])
# Edit configs/study.json BEFORE the first smoke run, not halfway through training.
# Epoch ceiling 500, patience 20; validation is 20% of the selected training pool.


Maximum epochs: 500 | Patience: 20
Datasets: ['ssdd', 'hrsid']
Experiment: study_v5_leakage_safe


In [7]:
import json

CONFIG = ROOT / "configs" / "study.json"
settings = json.loads(CONFIG.read_text())

print("Experiment:", settings["experiment"])
print("Seed:", settings["seed"])
print("Validation fraction:", settings["validation_fraction"])

assert settings["experiment"] == "study_v5_leakage_safe"
assert settings["seed"] == 42

print("Correct leakage-safe configuration loaded")

Experiment: study_v5_leakage_safe
Seed: 42
Validation fraction: 0.2
Correct leakage-safe configuration loaded


In [8]:
def run_final(model, dataset):
    command = [
        str(PYTHON),
        "-u",
        "-m",
        "sarstudy.runner",
        "--config", str(CONFIG),
        "--model", model,
        "--dataset", dataset,
    ]

    environment = os.environ.copy()
    environment["MPLBACKEND"] = "Agg"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    recent = deque(maxlen=200)

    important = (
        "ClearML Task:",
        "ClearML results page:",
        "Hashing ",
        "Excluded ",
        "Fingerprinting ",
        "Starting training",
        "epochs completed",
        "Predicting ",
        "Evaluating ",
        "Benchmarking ",
        "COMPLETE:",
        "EarlyStopping",
        "Traceback",
        "Error",
        "ERROR",
    )

    for line in process.stdout:
        recent.append(line)

        stripped = line.strip()
        epoch_summary = (
            stripped.startswith("Epoch ")
            and "train_loss=" in stripped
        )

        if epoch_summary or any(text in line for text in important):
            print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        print("\nLast output before failure:\n")
        print("".join(recent))
        raise RuntimeError(
            f"{model}/{dataset} failed with exit code {return_code}"
        )

    print(f"\n{model}/{dataset} finished successfully")

## 5. SAMSAR prerequisites

The external SAMSAR implementation must already exist in Drive. This downloads only the official SAM ViT-B initialization checkpoint, not old trained weights.

In [12]:
import urllib.request
from pathlib import Path

repo = Path(settings["samsar_repo"])

assert (
    repo / "samsar/modeling/build_samsar.py"
).is_file(), f"SAMSAR source not found: {repo}"

checkpoint = Path(settings["sam_checkpoint"])

if not checkpoint.is_file():
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    temporary = checkpoint.with_suffix(".download")

    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/"
        "sam_vit_b_01ec64.pth",
        temporary,
    )

    temporary.replace(checkpoint)

assert (
    checkpoint.stat().st_size > 100_000_000
), "SAM checkpoint looks incomplete"

print("SAMSAR prerequisites ready")
print("Repository:", repo)
print("Checkpoint:", checkpoint)

SAMSAR prerequisites ready
Repository: /content/drive/MyDrive/SAMSAR_Implementation
Checkpoint: /content/sam_vit_b_01ec64.pth


In [17]:
import json
from pathlib import Path

SAMSAR_CONFIG = ROOT / "configs" / "samsar_stable.json"

samsar_settings = settings.copy()
samsar_settings["experiment"] = "study_v6_samsar_stable"
samsar_settings["amp"] = False
samsar_settings["learning_rate"] = 0.00002
samsar_settings["models"] = ["samsar"]

SAMSAR_CONFIG.write_text(
    json.dumps(samsar_settings, indent=2)
)

CONFIG = SAMSAR_CONFIG
settings = samsar_settings

print("Experiment:", settings["experiment"])
print("AMP:", settings["amp"])
print("Learning rate:", settings["learning_rate"])

Experiment: study_v6_samsar_stable
AMP: False
Learning rate: 2e-05


## Run helpers

In [9]:
def run_one(model, dataset, smoke=True):
    command = [str(PYTHON), '-u', '-m', 'sarstudy.runner',
               '--config', str(CONFIG), '--model', model, '--dataset', dataset]
    if smoke:
        command.append('--smoke')
    else:
        marker = ROOT/'runs'/dataset/model/(settings['experiment']+'_smoke')/'evaluation_complete.json'
        assert marker.is_file(), f'Complete the two-epoch smoke run first: {model}/{dataset}'
    environment = os.environ.copy()
    environment['MPLBACKEND'] = 'Agg'
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=environment)
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'{model}/{dataset} failed with exit code {return_code}')


## Two-epoch verification

Run one cell at a time. Check that train_loss, val_loss and validation AP appear and that ClearML has the Loss / total graph. A smoke pass checks execution, not convergence.

In [ ]:
for dataset in settings['datasets']:
    run_one('yolo26n_seg', dataset, smoke=True)


ClearML Task: created new task id=3e710d1c9f324c3d9e79de0429961560
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/3e710d1c9f324c3d9e79de0429961560/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 8 images
Fingerprinting ssdd/val: 4 images
Fingerprinting ssdd/test: 4 images
{'train': 8, 'val': 4, 'test': 4}
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

New https://pypi.org/project/ultralytics/8.4.132 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic

In [ ]:
for dataset in settings['datasets']:
    run_one('yolo26s_seg', dataset, smoke=True)


ClearML Task: created new task id=a397ff592d634e5b9e3ec708377b0909
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/a397ff592d634e5b9e3ec708377b0909/output/log
{'train': 8, 'val': 4, 'test': 4}
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

New https://pypi.org/project/ultralytics/8.4.132 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=T

In [ ]:
for dataset in settings['datasets']:
    run_one('yolo26m_seg', dataset, smoke=True)


ClearML Task: created new task id=94aa7427f1e54646bb378b071e270fb2
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/94aa7427f1e54646bb378b071e270fb2/output/log
{'train': 8, 'val': 4, 'test': 4}

New https://pypi.org/project/ultralytics/8.4.133 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sar_ship_segmentation_cache/494583867358990e37b69207b135bae115b04222ad20f1734b8da75f4af3b8c1/ssdd/data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, 

In [ ]:
for dataset in ["ssdd", "hrsid"]:
    folder = (
        ROOT
        / "runs"
        / dataset
        / "yolo26m_seg"
        / "study_v5_leakage_safe"
    )

    print(
        dataset,
        "training:", (folder / "training_complete.json").is_file(),
        "evaluation:", (folder / "evaluation_complete.json").is_file(),
        "checkpoint:", (folder / "checkpoints" / "best.pt").is_file(),
        "failure:", (folder / "failure.txt").is_file(),
    )

ssdd training: True evaluation: True checkpoint: True failure: False
hrsid training: True evaluation: True checkpoint: True failure: False


In [ ]:
for dataset in settings['datasets']:
    run_one('rfdetr_seg', dataset, smoke=True)


ClearML Task: created new task id=f6722e9c5f6443c39d7c1d5326b5a67b
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/f6722e9c5f6443c39d7c1d5326b5a67b/output/log
{'train': 8, 'val': 4, 'test': 4}
[2026-08-31 02:32:50] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 02:32:50] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 02:32:51] [INFO] rf-detr - Downloading pretrained weights for rf-detr-seg-nano.pt

rf-detr-seg-nano.pt:   0%|          | 0.00/128M [00:00<?, ?iB/s]
rf-detr-seg-nano.pt:   0%|          | 183k/128M [00:00<01:16, 1.76MiB/s]
rf-detr-seg-nano.pt:   0%|          | 356k/128M [00:00<01:25, 1.57MiB/s]
rf-detr-seg-nano.pt:   0%|          | 5

In [18]:
for dataset in settings['datasets']:
    run_one('samsar', dataset, smoke=True)


ClearML Task: created new task id=bf8df6c8b49341cf944190e09bab07d9
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/bf8df6c8b49341cf944190e09bab07d9/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 8 images
Fingerprinting ssdd/val: 4 images
Fingerprinting ssdd/test: 4 images
{'train': 8, 'val': 4, 'test': 4}
[SAMSAR] loaded 314/314 pre-trained SAM tensors (114 new-module tensors initialized from scratch, 0 checkpoint tensors unused)
[SAMSAR] parameters: 132.0M total, 42.3M trainable
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.00s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.563
 Av

In [20]:
for dataset in settings['datasets']:
    run_one('ship_contour', dataset, smoke=True)


ClearML Task: created new task id=c8b6444ca303453eb4c7f4e1e4de0e5d
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/c8b6444ca303453eb4c7f4e1e4de0e5d/output/log
{'train': 8, 'val': 4, 'test': 4}
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100

## Long runs — only after checking smoke results

Enable this deliberately. Each model/dataset has its own history and checkpoint folder. If Colab disconnects, repeat setup and rerun the same cell; saved state is resumed. Do not run two copies of the same experiment simultaneously.

In [15]:
APPROVE_LONG_RUNS = True  # Change to True only after checking smoke logs and curves.


In [ ]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('yolo26n_seg', dataset, smoke=False)


Streaming output truncated to the last 5000 lines.
     68/500      2.05G      1.379      1.499     0.8884   0.003151     0.6312          5        512: 37% ━━━━╸─────── 189/500 24.3it/s 7.1s<12.8s
     68/500      2.05G      1.378      1.509     0.8881   0.003137     0.6335          2        512: 38% ━━━━╸─────── 192/500 24.6it/s 7.2s<12.5s
     68/500      2.05G      1.376      1.501     0.8869   0.003123     0.6291          3        512: 39% ━━━━╸─────── 195/500 24.7it/s 7.4s<12.4s
     68/500      2.05G      1.372      1.496     0.8816   0.003118     0.6265          9        512: 39% ━━━━╸─────── 198/500 24.8it/s 7.5s<12.2s
     68/500      2.05G      1.371      1.501     0.8778     0.0031     0.6235         13        512: 40% ━━━━╸─────── 201/500 24.8it/s 7.6s<12.1s
     68/500      2.05G      1.369      1.503     0.8765   0.003101     0.6227          1        512: 40% ━━━━╸─────── 204/500 24.6it/s 7.7s<12.0s
     68/500      2.05G      1.364      1.493     0.8729   0.003085     0.

In [ ]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('yolo26s_seg', dataset, smoke=False)


ClearML Task: created new task id=35cf68fbd40c48f38de8a674daf22de4
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/35cf68fbd40c48f38de8a674daf22de4/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 733, 'val': 183, 'test': 232}
New https://pypi.org/project/ultralytics/8.4.132 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sar_ship_segmentation_cache/38c4a04285d388f697dc3b42197c782ee5c4416dbc9f0dc59e23836b96e3cc0a

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/SAR_Ship_Segmentation_Study")

folder = (
    ROOT
    / "runs"
    / "ssdd"
    / "yolo26s_seg"
    / "study_v5_leakage_safe"
)

print("Run folder:", folder)
print("Training complete:", (folder / "training_complete.json").is_file())
print("Evaluation complete:", (folder / "evaluation_complete.json").is_file())
print("Best checkpoint:", (folder / "checkpoints" / "best.pt").is_file())
print("Last checkpoint:", (folder / "checkpoints" / "last.pt").is_file())
print("Failure file:", (folder / "failure.txt").is_file())

failure = folder / "failure.txt"

if failure.is_file():
    print("\nLAST PART OF FAILURE:\n")
    print("\n".join(failure.read_text().splitlines()[-40:]))

Run folder: /content/drive/MyDrive/SAR_Ship_Segmentation_Study/runs/ssdd/yolo26s_seg/study_v5_leakage_safe
Training complete: True
Evaluation complete: True
Best checkpoint: True
Last checkpoint: True
Failure file: False


In [ ]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('yolo26m_seg', dataset, smoke=False)


ClearML Task: created new task id=c62a40113c5a459283866ab12bfbc5eb
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/c62a40113c5a459283866ab12bfbc5eb/output/log
{'train': 733, 'val': 183, 'test': 232}
New https://pypi.org/project/ultralytics/8.4.133 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.7.1+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sar_ship_segmentation_cache/38c4a04285d388f697dc3b42197c782ee5c4416dbc9f0dc59e23836b96e3cc0a/ssdd/data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=N

In [ ]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('rfdetr_seg', dataset, smoke=False)


ClearML Task: created new task id=8dec62584c2b474986f0fc9dfb252057
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/8dec62584c2b474986f0fc9dfb252057/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 733, 'val': 183, 'test': 232}
[2026-08-31 02:43:02] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 02:43:02] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 02:43:02] [INFO] rf-detr - File rf-detr-seg-nano.pt already exists with correct MD5 hash.
[2026-08-31 02:43:02] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 1. The detection head will be re-init

In [10]:
run_one("rfdetr_seg", "ssdd", smoke=False)

ClearML Task: created new task id=5bee6626587043029bf8553d0f8f3932
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/5bee6626587043029bf8553d0f8f3932/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 733, 'val': 183, 'test': 232}
[2026-08-31 06:09:23] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 06:09:23] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t

In [11]:
run_one("rfdetr_seg", "hrsid", smoke=False)

ClearML Task: created new task id=1f8bc6a5452f4be5ababcb19e80e5fe4
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/1f8bc6a5452f4be5ababcb19e80e5fe4/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 1000, 'val': 250, 'test': 500}

                                           0% | 0.00/32.85 MB [00:00<?, ?MB/s]: 
█████████▍                       30% | 10.00/32.85 MB [00:00<00:00, 55.53MB/s]: [2026-08-31 06:42:17] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-31 06:42:17] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.

██████████████████▊              61% | 20.00/32.85 MB [00:00<00:00, 36.

In [19]:
for dataset in settings["datasets"]:
    print("\n" + "=" * 70)
    print("Running stable SAMSAR on:", dataset)
    print("=" * 70)

    run_one("samsar", dataset, smoke=False)


Running stable SAMSAR on: ssdd
ClearML Task: created new task id=dc8a0a6990084ff1847d99254da0bef4
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/dc8a0a6990084ff1847d99254da0bef4/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 733 images
Fingerprinting ssdd/val: 183 images
Fingerprinting ssdd/test: 232 images
{'train': 733, 'val': 183, 'test': 232}
[SAMSAR] loaded 314/314 pre-trained SAM tensors (114 new-module tensors initialized from scratch, 0 checkpoint tensors unused)
[SAMSAR] parameters: 132.0M total, 42.3M trainable
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.03s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.

In [21]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('ship_contour', dataset, smoke=False)


Streaming output truncated to the last 5000 lines.
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.212
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.444
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.482
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.507
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.406
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.100
Epoch 91: train_loss=37.46237 val_loss=386.76734 val_mask_AP=31.675
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *segm*
DONE (t=0.30s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.288
 Average Precision  (AP) @

## Check durable outputs

In [22]:
for dataset in settings['datasets']:
    for model in settings['models']:
        folder = ROOT/'runs'/dataset/model/settings['experiment']
        print(dataset, model, 'training:', (folder/'training_complete.json').is_file(),
              'evaluation:', (folder/'evaluation_complete.json').is_file())


ssdd samsar training: True evaluation: True
hrsid samsar training: True evaluation: True
